# CS 195: Natural Language Processing
## PyTorch Embeddings

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ericmanley/s26-CS195NLP/blob/main/F4_2_PyTorchEmbeddings.ipynb)


## References

[SLP: Embeddings, Chapter 5 of Speech and Language Processing by Daniel Jurafsky & James H. Martin](https://web.stanford.edu/~jurafsky/slp3/5.pdf)

[Word2Vec Tutorial - The Skip-Gram Model by Chris McCormick](http://mccormickml.com/2016/04/19/word2vec-tutorial-the-skip-gram-model/)

[Word2Vec - Negative Sampling made easy by Munesh Lakhey](https://medium.com/@mnshonco/word2vec-negative-sampling-made-easy-9a587cb4695f)

[PyTorch `nn.Embedding` documentation](https://pytorch.org/docs/stable/generated/torch.nn.Embedding.html)

[PyTorch `torch.nn.functional.one_hot` documentation](https://pytorch.org/docs/stable/generated/torch.nn.functional.one_hot.html)


In [ ]:
#import sys
#!{sys.executable} -m pip install datasets torch scikit-learn transformers tokenizers

## Reorganization of where we left off from last time

In [1]:
from tokenizers import Tokenizer, models, trainers, pre_tokenizers
import random 
import torch
import torch.nn.functional as F
import torch.nn as nn
import torch.optim as optim

def train_tokenizer(sentences, vocabulary_size=200):
    tokenizer = Tokenizer(models.BPE())
    tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
    trainer = trainers.BpeTrainer(vocab_size=vocabulary_size)

    tokenizer.train_from_iterator(sentences, trainer)
    return tokenizer


def make_skipgrams(sequence, vocabulary_size, window_size=3):
    couples = []
    labels = []

    for i in range(len(sequence)):
        target = sequence[i]
        left = max(0, i - window_size)
        right = min(len(sequence), i + window_size + 1)

        for j in range(left, right):
            if i != j:
                context = sequence[j]

                # positive pair
                couples.append((target, context))
                labels.append(1)

                # generate a random negative pair (in real life, you might want to generate several negative pairs per positive pair)
                negative_context = random.randint(1, vocabulary_size - 1)
                # TODO: we should probably check to make sure this isn't actually a positive pair
                couples.append((target, negative_context))
                labels.append(0)

    return [couples, labels]

def make_skipgrams_batch(tokenized_sentences, vocabulary_size, window_size=3):
    all_couples = []
    all_labels = []
    for tokenized_sentence in tokenized_sentences:
        couples, labels = make_skipgrams(tokenized_sentence, vocabulary_size, window_size)
        all_couples.extend(couples)
        all_labels.extend(labels)

    return [all_couples, all_labels]


def prepare_one_hot_inputs(couples, labels, vocabulary_size):
    inputs = []
    for target_word, context_word in couples:
        target_one_hot = F.one_hot(torch.tensor(target_word), num_classes=vocabulary_size)
        context_one_hot = F.one_hot(torch.tensor(context_word), num_classes=vocabulary_size)
        inputs.append(torch.cat([target_one_hot, context_one_hot]))

    inputs_array = torch.stack(inputs).float()
    labels_array = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

    return inputs_array, labels_array


def train_embedding_model(inputs_array, labels_array, vocabulary_size):

    embedding_model = nn.Sequential(
        nn.Linear(vocabulary_size * 2, 50),
        nn.ReLU(),
        nn.Linear(50, 1)
    )

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.0001)

    for epoch in range(20000):
        logits = embedding_model(inputs_array)
        loss = loss_fn(logits, labels_array)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 1000 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids
    first_layer_weights = embedding_model[0].weight.detach()
    return first_layer_weights[:, word_ids[0]] # use first subword token for this demo



In [2]:
sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print("Here's an example of some tokens:", tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)

inputs_array, labels_array = prepare_one_hot_inputs(couples, labels, tokenizer.get_vocab_size())

embedding_model = train_embedding_model(inputs_array, labels_array, vocabulary_size=tokenizer.get_vocab_size())





Here's an example of some tokens: ['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 1000, loss=0.4142
epoch 2000, loss=0.2746
epoch 3000, loss=0.2094
epoch 4000, loss=0.1603
epoch 5000, loss=0.1216
epoch 6000, loss=0.0936
epoch 7000, loss=0.0759
epoch 8000, loss=0.0649
epoch 9000, loss=0.0586
epoch 10000, loss=0.0549
epoch 11000, loss=0.0527
epoch 12000, loss=0.0515
epoch 13000, loss=0.0507
epoch 14000, loss=0.0503
epoch 15000, loss=0.0500
epoch 16000, loss=0.0499
epoch 17000, loss=0.0498
epoch 18000, loss=0.0497
epoch 19000, loss=0.0497
epoch 20000, loss=0.0496


In [3]:

cats_embedding = get_embedding("cats", tokenizer, embedding_model)
dogs_embedding = get_embedding("dogs", tokenizer, embedding_model)
ocean_embedding = get_embedding("ocean", tokenizer, embedding_model)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)


tensor([-0.5463,  0.7711,  0.2590,  0.9406,  0.0172,  0.7669,  0.6977,  0.9133,
         0.9967,  0.6733,  1.0261,  0.9750,  0.7895,  0.8236, -0.5990, -1.0094,
         0.9644,  0.1287, -1.0763,  0.9244,  0.7460,  0.0628, -0.9943,  0.9487,
         0.6258, -0.8846, -0.8385,  0.9074,  0.6680,  0.9396,  1.0120, -0.1871,
        -1.0446,  0.4363,  0.9584,  0.8368,  0.8224,  0.9085, -0.7437,  0.9911,
         1.0755,  0.9369,  0.7378,  1.0163,  0.4969,  0.7209,  0.8822, -1.0229,
         0.8660,  1.0427])
tensor([ 0.2288,  0.7714, -0.0931,  0.9350,  0.8756,  0.9482, -0.6110, -0.2883,
        -0.2576,  0.8710, -0.9711, -0.1086,  0.8396,  0.9895,  0.7025,  1.0559,
         0.9872,  0.8649, -0.0712,  0.9244,  0.5102,  0.5436,  1.0708,  0.5377,
         0.9565,  0.7234,  0.7143,  0.6479,  0.9202, -0.2437,  1.0120,  0.5720,
        -0.1609,  0.4086, -1.1234,  0.8368,  0.9589,  0.9102,  1.0822, -0.4993,
        -0.4200, -0.1102,  0.8595, -1.0616,  1.0538,  0.2558,  0.7147, -0.3011,
        -1.14

## Dataset for today

AG News dataset
* short news articles
* four classes: World, Sports, Business, Sci/Tech

https://huggingface.co/datasets/fancyzhx/ag_news


In [4]:
from datasets import load_dataset
data = load_dataset("ag_news")

print(data["train"]["text"][0])

# 0 is World
# 1 is Sports
# 2 is Business
# 3 is Sci/Tech
print(data["train"]["label"][0])

Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
2


## Group Exercise

Try creating word embeddings for the AG News dataset. *Note that you will only be able to do a very small subset with this code. Start with 10 examples and work your way up*

* How big of a vocabulary did you need to use to get reasonable tokens for this data?
* Once you use more examples, you'll start to notice that you're going to run out of memory and the kernel will crash. What do you think the issue is? Can you think of a way to do the same idea but keep less data in memory at any given time?



## The PyTorch Embedding Layer
PyTorch provides an `nn.Embedding` layer, which is a linear layer with a lookup table. It allows you to input a token id and look up a row vector (akin to the linear node associated with that id).

It's mathematically equivalent to the one-hot encoding + `nn.Linear` layer we saw before, but it much more memory efficient - we don't have to waste space storing all of those 0s. 

Let's try the same experiment as before but using the `nn.Embedding` layer. 

Note that for this one, we take the dot product 
* (multiply corresponding elements in the vector, add them all up) of the *target* and *context* vectors rather than concatenating their one-hot representations
* product of each dimension contributes towards agreement
* large dot product -> high agreement between the vectors
* `BCEWithLogitsLoss` pushes positive pairs to higher output logits and negative pairs to lower ones
* words appearing with similar contexts get pushed in similar directions


In [5]:
def train_embedding_model_v2(couples, labels, vocabulary_size):

    embedding_model = nn.Embedding(vocabulary_size, 50)

    loss_fn = nn.BCEWithLogitsLoss() # don't forget - this includes the sigmoid squashing function 
    optimizer = optim.Adam(embedding_model.parameters(), lr=0.001)

    pair_ids = torch.tensor(couples, dtype=torch.long)                # [N, 2]
    labels_tensor = torch.tensor(labels, dtype=torch.float32).unsqueeze(1) # [N, 1]

    for epoch in range(2000):
        target_ids = pair_ids[:, 0]
        context_ids = pair_ids[:, 1]

        target_embeddings = embedding_model(target_ids)
        context_embeddings = embedding_model(context_ids)

        # dot product between target and context embeddings to get a single logit for each pair
        logits = (target_embeddings * context_embeddings).sum(dim=1, keepdim=True)
        loss = loss_fn(logits, labels_tensor )

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if (epoch + 1) % 100 == 0:
            print(f"epoch {epoch+1}, loss={loss.item():.4f}")

    return embedding_model

def get_embedding_v2(word, tokenizer, embedding_model):
    word_ids = tokenizer.encode(word).ids[0] # use first subword token for this demo
    word_weights = embedding_model.weight.detach()
    return word_weights[word_ids]

sentences = [
    "I adopted some dogs from the animal shelter",
    "don't you know that dogs and cats both like scritches",
    "are cats or dogs your favorite animal",
    "I have heard that dogs can be obedient",
    "I have heard that cats can be independent",
    "sharks live in the ocean",
    "many birds fly to get around",
    "dogs and cats are common household pets",
    "cats and dogs both need food and water",
    "my dog and my cat play together",
    "cats and dogs can live in the same home",
    "the puppy and kitten were adopted together",
    "fish swim in the ocean",
    "whales and sharks live in the ocean",
    "boats travel across the ocean",
    "the ocean water is deep and salty",
    "coral reefs are in the ocean"
]

tokenizer = train_tokenizer(sentences, vocabulary_size=200)

tokenized_sentences = []
for sentence in sentences:
    ids = tokenizer.encode(sentence).ids
    tokens = tokenizer.encode(sentence).tokens
    tokenized_sentences.append(ids)

print(tokens)

couples, labels = make_skipgrams_batch(tokenized_sentences, vocabulary_size=tokenizer.get_vocab_size(), window_size=2)
embedding_model_v2 = train_embedding_model_v2(couples, labels, vocabulary_size=tokenizer.get_vocab_size())




['coral', 'reefs', 'are', 'in', 'the', 'ocean']
epoch 100, loss=1.4871
epoch 200, loss=0.6755
epoch 300, loss=0.3430
epoch 400, loss=0.2218
epoch 500, loss=0.1758
epoch 600, loss=0.1516
epoch 700, loss=0.1352
epoch 800, loss=0.1225
epoch 900, loss=0.1123
epoch 1000, loss=0.1038
epoch 1100, loss=0.0968
epoch 1200, loss=0.0909
epoch 1300, loss=0.0860
epoch 1400, loss=0.0818
epoch 1500, loss=0.0784
epoch 1600, loss=0.0755
epoch 1700, loss=0.0730
epoch 1800, loss=0.0710
epoch 1900, loss=0.0694
epoch 2000, loss=0.0680


In [6]:

cats_embedding = get_embedding_v2("cats", tokenizer, embedding_model_v2)
dogs_embedding = get_embedding_v2("dogs", tokenizer, embedding_model_v2)
ocean_embedding = get_embedding_v2("ocean", tokenizer, embedding_model_v2)

print(cats_embedding)
print(dogs_embedding)

# distance between dogs and cats should be smaller than distance between dogs and ocean
print(torch.sum((dogs_embedding - cats_embedding) ** 2).item())
print(torch.sum((dogs_embedding - ocean_embedding) ** 2).item())

# let's also look at cosine similarity, which is a common way to measure similarity between vectors that ignores differences in magnitude
# a negative cosine similarity means the vectors are pointing in opposite directions, a positive cosine similarity means they are pointing in the same direction, and a cosine similarity close to 0 means they are orthogonal (i.e. not similar at all)
dogs_cats_similarity = F.cosine_similarity(dogs_embedding, cats_embedding, dim=0).item()
dogs_ocean_similarity = F.cosine_similarity(dogs_embedding, ocean_embedding, dim=0).item()
print("dogs, cats similarity", dogs_cats_similarity)
print("dogs, ocean similarity", dogs_ocean_similarity)

tensor([ 1.2610, -0.4969, -0.2546,  0.6453,  0.0985, -0.1142,  0.1869,  1.8520,
         2.3872, -2.0876,  2.3715,  0.9325,  0.2839,  1.6071,  0.6645, -0.4938,
        -1.0436, -1.0109, -0.6181, -0.1205, -1.2305, -0.1990,  0.6604,  0.0326,
         0.4242, -0.0391,  0.4482,  0.5510, -0.2467, -1.1311,  1.1067, -0.1370,
         0.3926,  0.8683,  0.8562, -1.3429, -0.6061,  0.4300, -0.5783, -1.8025,
        -0.7380,  0.7615, -1.0096, -1.5513, -0.0977,  1.5166, -0.9673, -1.5305,
         1.0409, -0.8255])
tensor([-0.8665,  0.4322, -1.3244,  0.1674, -1.5157, -1.0971, -0.4654, -0.8156,
         1.4183,  0.1328, -0.9938, -1.1456, -1.0086,  1.1142, -1.2966, -0.4692,
         1.2580, -0.9386, -0.8587, -0.5227, -1.3987, -0.0449,  0.4185, -1.5820,
        -0.9602, -1.0813, -0.2690, -0.7146,  0.1780, -0.2190,  0.4616,  1.0285,
        -1.2056, -0.5985,  2.2349, -0.8474,  1.2459, -0.7268, -0.5561, -0.7504,
         1.2027,  0.9396, -1.2252, -0.5611,  0.2022,  1.8992,  1.0130, -0.9164,
         0.82

## Applied Exploration

Create word embeddings for a larger portion of the AG News dataset, say 5000 texts

Show some example word embeddings for some words that appear in the dataset (*cats* and *dogs* may not be good examples for this one)

Describe your results and reflect on them
* How big of a vocabulary did you need to use?
* What learning rate and number of training epochs do you think are appropriate? Why?
* How could you go about figuring out if these embeddings are useful?

## Adding an Embedding layer to your model for other learning tasks

First, let's prepare the data
* We could use the same kind of tokenizer, or just use an existing one - let's go back to doing it with a pretrained Hugging Face tokenizer


In [7]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM

data = load_dataset("ag_news")

train_texts = data["train"]["text"]
test_texts  = data["test"]["text"]
train_labels = data["train"]["label"]
test_labels = data["test"]["label"]

hf_tokenizer = AutoTokenizer.from_pretrained("HuggingFaceTB/SmolLM2-135M")
hf_tokenizer.add_special_tokens({'pad_token': '[PAD]'}) # for some reason this tokenizer doesn't have a pad token by default, but we need one to be able to batch our inputs together, so we'll just add a new special token for padding

tokenized_train_texts = hf_tokenizer(list(train_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")
tokenized_test_texts = hf_tokenizer(list(test_texts), truncation=True, padding=True, max_length=128, return_tensors="pt")

X_train = tokenized_train_texts["input_ids"]
X_test = tokenized_test_texts["input_ids"]

y_train = torch.tensor(np.array(train_labels), dtype=torch.long)
y_test = torch.tensor(np.array(test_labels), dtype=torch.long)


## Model with an Embedding layer

We'll set up the embedding layer and sequential model as before

The optimizer needs to have a concatenation of the parameters for both parts

In [8]:
embedding = nn.Embedding(len(hf_tokenizer), 50) # len(hf_tokenizer) is the vocab size including the new padding token
classifier = nn.Sequential(
    nn.Linear(50, 100),
    nn.ReLU(),
    nn.Linear(100, 4)
)

loss_fn = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(embedding.parameters()) + list(classifier.parameters()), lr=0.01)

## Training Loop with Embedding layer

The input goes into the embedding layer
* returns an embedding for each words, so we aggregate them with the *mean* to get an embedding for the whole training example

In [9]:


for epoch in range(100):
    optimizer.zero_grad()

    emb = embedding(X_train) # [batch_size, seq_len, embedding_dim]
    pooled_emb = emb.mean(dim=1) # simple way to get a single vector [batch_size, embedding_dim] for the whole sequence - just average the token embeddings
    logits = classifier(pooled_emb)
    loss = loss_fn(logits, y_train)

    loss.backward()
    optimizer.step()

    if epoch % 10 == 0:
        print(f"Epoch {epoch+1}, loss = {loss.item():.4f}")



Epoch 1, loss = 1.3864
Epoch 11, loss = 1.3261
Epoch 21, loss = 1.0533
Epoch 31, loss = 0.6228
Epoch 41, loss = 0.3929
Epoch 51, loss = 0.2925
Epoch 61, loss = 0.2287
Epoch 71, loss = 0.1849
Epoch 81, loss = 0.1528
Epoch 91, loss = 0.1272


## Evaluating works similarly

In [10]:
with torch.no_grad():
    emb = embedding(X_test)
    pooled_emb = emb.mean(dim=1)
    logits = classifier(pooled_emb)
    predicted_labels = logits.argmax(dim=1)
    accuracy = (predicted_labels == y_test).float().mean()

print("Accuracy:", accuracy.item())

Accuracy: 0.902236819267273


## Applied Exploration

Perform a text classification experiment with another classification data set
* Try different embedding vector lengths (other than just 50 as we did here)
* Experiment with different neural network structures, learning rates, and number of epochs

Report your results and reflect on them
* Describe your dataset
* Describe what you did
* Report the results you observed
* Discuss any interesting insights